# NB07: Reusable Cultivation-Bias Diagnostic

Package the per-function cultivation-coverage analysis into a reusable module
that can be applied to any cultured-vs-MAG comparison in BERDL.

**Requires**: NB04 complete (to validate against known results)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path

DATA_DIR = Path('../data')

## 1. Cultivation Bias Diagnostic Function

In [ ]:
def cultivation_bias_diagnostic(
    cultured_ko: pd.DataFrame,
    mag_ko: pd.DataFrame,
    markers: dict = None,
    fdr_threshold: float = 0.05,
    pseudocount: float = 0.001
) -> dict:
    """
    Compute per-KO cultivation coverage comparing cultured genomes vs MAGs.
    
    Parameters
    ----------
    cultured_ko : DataFrame with columns [genome_id, ko_id]
    mag_ko : DataFrame with columns [genome_id, ko_id]
    markers : dict mapping category name -> list of KO IDs (optional)
    fdr_threshold : significance cutoff after BH correction
    pseudocount : added to prevalence fractions before log2 ratio
    
    Returns
    -------
    dict with keys: 'coverage' (full DataFrame), 'markers' (marker summary),
                    'summary' (dict of counts)
    """
    n_cult = cultured_ko['genome_id'].nunique()
    n_mag = mag_ko['genome_id'].nunique()
    
    cult_prev = cultured_ko.groupby('ko_id')['genome_id'].nunique().rename('n_cult')
    mag_prev = mag_ko.groupby('ko_id')['genome_id'].nunique().rename('n_mag')
    
    all_kos = sorted(set(cult_prev.index) | set(mag_prev.index))
    cov = pd.DataFrame(index=all_kos)
    cov['n_cult'] = cult_prev.reindex(cov.index).fillna(0).astype(int)
    cov['n_mag'] = mag_prev.reindex(cov.index).fillna(0).astype(int)
    cov['frac_cult'] = cov['n_cult'] / n_cult
    cov['frac_mag'] = cov['n_mag'] / n_mag
    cov['log2_ratio'] = np.log2(
        (cov['frac_cult'] + pseudocount) / (cov['frac_mag'] + pseudocount)
    )
    
    pvals = []
    for ko_id in cov.index:
        a = cov.loc[ko_id, 'n_cult']
        b = n_cult - a
        c = cov.loc[ko_id, 'n_mag']
        d = n_mag - c
        _, p = stats.fisher_exact([[a, b], [c, d]])
        pvals.append(p)
    
    cov['pval'] = pvals
    reject, qvals, _, _ = multipletests(cov['pval'], method='fdr_bh')
    cov['qval'] = qvals
    cov['significant'] = reject
    
    sig = cov[cov['significant']]
    summary = {
        'n_cultured_genomes': n_cult,
        'n_mag_genomes': n_mag,
        'n_kos_total': len(cov),
        'n_significant': len(sig),
        'n_enriched_cultured': len(sig[sig['log2_ratio'] > 0]),
        'n_depleted_cultured': len(sig[sig['log2_ratio'] < 0]),
        'n_cultured_only': (cov['n_mag'] == 0).sum(),
        'n_mag_only': (cov['n_cult'] == 0).sum(),
    }
    
    result = {'coverage': cov, 'summary': summary}
    
    if markers:
        marker_rows = []
        for cat, kos in markers.items():
            for ko in kos:
                if ko in cov.index:
                    row = cov.loc[ko]
                    marker_rows.append({
                        'category': cat, 'ko_id': ko,
                        'frac_cult': row['frac_cult'], 'frac_mag': row['frac_mag'],
                        'log2_ratio': row['log2_ratio'], 'qval': row['qval'],
                        'significant': row['significant']
                    })
        result['markers'] = pd.DataFrame(marker_rows)
    
    return result

## 2. Validate Against NB04 Results

In [ ]:
cult_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
mag_ko_path = DATA_DIR / 'mag_ko_profiles.tsv'

if mag_ko_path.exists():
    mag_ko = pd.read_csv(mag_ko_path, sep='\t')
    
    markers = {
        'Wood-Ljungdahl': ['K00198', 'K00194', 'K00197', 'K15022', 'K15023'],
        'NiFe-hydrogenase': ['K06281', 'K06282'],
        'Sulfate reduction': ['K11180', 'K11181', 'K00394', 'K00395', 'K00958'],
        'Methanogenesis': ['K00399', 'K00401', 'K00402'],
        'Aerobic respiration': ['K02274', 'K02275', 'K02276'],
    }
    
    result = cultivation_bias_diagnostic(cult_ko, mag_ko, markers=markers)
    
    print('Summary:', result['summary'])
    print('\nMarker results:')
    print(result['markers'].to_string(index=False))
    
    # Compare with NB04 output
    nb04 = pd.read_csv(DATA_DIR / 'ko_cultivation_coverage_full.tsv', sep='\t', index_col=0)
    shared = result['coverage'].index.intersection(nb04.index)
    corr = np.corrcoef(result['coverage'].loc[shared, 'log2_ratio'], 
                       nb04.loc[shared, 'log2_ratio'])[0, 1]
    print(f'\nCorrelation with NB04 results: r={corr:.4f}')
else:
    print('MAG KO profiles not yet available — run NB02 first')
    print('Diagnostic function is ready for use once MAG annotations are complete')

## 3. Export Module

The `cultivation_bias_diagnostic()` function above can be imported from
`src/cultivation_bias.py` for use in other projects.

In [ ]:
# Write the function to a standalone module
import inspect

module_path = Path('../src')
module_path.mkdir(parents=True, exist_ok=True)

module_code = '''
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

''' + inspect.getsource(cultivation_bias_diagnostic)

with open(module_path / 'cultivation_bias.py', 'w') as f:
    f.write(module_code)

print(f'Module written to {module_path / "cultivation_bias.py"}')